# 02 — Clean and align the Belgian solar data

This notebook makes the deterministic cleaning rules visible and reproducible. It starts from the frozen raw **Elia** generation CSV and **Open-Meteo GFS** weather responses, validates their provenance, cleans each source, aligns them hourly, and reconstructs the rows eligible for modeling.

Cleaning principles:

- Raw files are read-only and are never overwritten.
- Timestamps are parsed explicitly and normalized to UTC.
- Missing targets are **not** filled with zero or interpolated.
- An hourly target requires four non-null 15-minute generation values.
- Source cloud-cover artifacts at -1% or 101% are flagged and clipped to 0–100%.
- Missing weather is preserved until the documented complete-case filter.
- Weather valid time `T+1` is aligned with generation interval `[T, T+1)`.
- No model is fitted, and the sealed 2025 test target is not summarized.

The production implementation remains `scripts/prepare_training_data.py`; this notebook calls the same tested functions rather than creating a second cleaning pipeline.

## 1. Setup and required files

In [1]:
from __future__ import annotations

import hashlib
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)


def find_repository_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Start Jupyter inside the Solar_forecast repository."
    )


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


ROOT = find_repository_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.prepare_training_data import (
    ELIA_COLUMNS,
    add_generation_lags,
    add_temporal_features,
    assign_splits,
    load_elia_hourly,
    load_weather_wide,
)

ELIA_DATA_PATH = ROOT / "data/raw/elia_generation/belgium_2024_2025.csv"
ELIA_MANIFEST_PATH = ROOT / "data/raw/elia_generation/source_manifest.json"
WEATHER_ROOT = ROOT / "data/raw/belgium_weather_archive"
WEATHER_MANIFEST_PATH = WEATHER_ROOT / "download_manifest.json"
GRID_CONFIG_PATH = ROOT / "config/belgium_weather_grid.json"
TRAINING_CONFIG_PATH = ROOT / "config/training_dataset.json"
PREPARED_ROOT = ROOT / "data/processed/model_training"
DATASET_MANIFEST_PATH = PREPARED_ROOT / "dataset_manifest.json"

required_paths = [
    ELIA_DATA_PATH,
    ELIA_MANIFEST_PATH,
    WEATHER_MANIFEST_PATH,
    GRID_CONFIG_PATH,
    TRAINING_CONFIG_PATH,
    DATASET_MANIFEST_PATH,
]
missing_paths = [path for path in required_paths if not path.is_file()]
if missing_paths:
    missing_text = "\n".join(f"- {path.relative_to(ROOT)}" for path in missing_paths)
    raise FileNotFoundError(
        "Required data is missing:\n"
        f"{missing_text}\n\n"
        "Run the documented download and preparation commands from the repository root."
    )

print(f"Repository root: {ROOT}")
print(f"pandas version: {pd.__version__}")

Repository root: /Users/akshith/Desktop/Solar_forecast
pandas version: 3.1.1


## 2. Verify source provenance

The manifests freeze source URLs, schemas, byte sizes, and SHA-256 hashes. Cleaning stops if the retained Elia file or weather manifest differs from the reviewed inputs. Individual weather-response hashes are verified by the weather loader in section 4.

In [2]:
with ELIA_MANIFEST_PATH.open(encoding="utf-8") as source:
    elia_manifest = json.load(source)
with WEATHER_MANIFEST_PATH.open(encoding="utf-8") as source:
    weather_manifest = json.load(source)
with GRID_CONFIG_PATH.open(encoding="utf-8") as source:
    grid_config = json.load(source)
with TRAINING_CONFIG_PATH.open(encoding="utf-8") as source:
    training_config = json.load(source)
with DATASET_MANIFEST_PATH.open(encoding="utf-8") as source:
    dataset_manifest = json.load(source)

source_checks = pd.DataFrame(
    [
        {
            "source": "Elia generation CSV",
            "path": str(ELIA_DATA_PATH.relative_to(ROOT)),
            "bytes_match": ELIA_DATA_PATH.stat().st_size
            == elia_manifest["file"]["bytes"],
            "hash_matches_source_manifest": sha256_file(ELIA_DATA_PATH)
            == elia_manifest["file"]["sha256"],
            "hash_matches_dataset_manifest": sha256_file(ELIA_DATA_PATH)
            == dataset_manifest["audit"]["elia"]["sha256"],
        },
        {
            "source": "Open-Meteo weather manifest",
            "path": str(WEATHER_MANIFEST_PATH.relative_to(ROOT)),
            "bytes_match": True,
            "hash_matches_source_manifest": True,
            "hash_matches_dataset_manifest": sha256_file(WEATHER_MANIFEST_PATH)
            == dataset_manifest["audit"]["weather"]["weather_manifest_sha256"],
        },
    ]
)
check_columns = [
    "bytes_match",
    "hash_matches_source_manifest",
    "hash_matches_dataset_manifest",
]
if not source_checks[check_columns].all().all():
    raise ValueError("A retained source differs from its reviewed manifest.")

source_checks

,source,path,bytes_match,hash_matches_source_manifest,hash_matches_dataset_manifest
0,Elia generation CSV,data/raw/elia_generation/belgium_2024_2025.csv,True,True,True
1,Open-Meteo weather manifest,data/raw/belgium_weather_archive/download_mani...,True,True,True


## 3. Clean the Elia generation target

The source contains one row per 15-minute interval. The target field is `measured`, an Elia measured/upscaled regional PV estimate in MW. Loading is explicit: semicolon delimiter, reviewed columns, ISO-8601 timestamps, UTC normalization, and numeric conversion.

In [3]:
raw_generation = pd.read_csv(
    ELIA_DATA_PATH,
    sep=";",
    encoding="utf-8",
    dtype={
        "datetime": "string",
        "resolutioncode": "string",
        "region": "string",
    },
    low_memory=False,
)
missing_columns = sorted(set(ELIA_COLUMNS) - set(raw_generation.columns))
if missing_columns:
    raise ValueError(f"Elia source is missing required columns: {missing_columns}")

raw_generation = raw_generation.loc[:, ELIA_COLUMNS].copy()
raw_generation["timestamp"] = pd.to_datetime(
    raw_generation["datetime"], format="ISO8601", errors="raise", utc=True
)
for column in ["measured", "dayaheadforecast", "monitoredcapacity"]:
    raw_generation[column] = pd.to_numeric(raw_generation[column], errors="raise")
raw_generation = raw_generation.sort_values("timestamp", kind="stable").reset_index(drop=True)

generation_audit = pd.Series(
    {
        "rows": len(raw_generation),
        "columns": len(raw_generation.columns),
        "first_timestamp_utc": raw_generation["timestamp"].min(),
        "last_timestamp_utc": raw_generation["timestamp"].max(),
        "duplicate_timestamps": int(raw_generation["timestamp"].duplicated().sum()),
        "unexpected_15_minute_steps": int(
            raw_generation["timestamp"].diff().dropna().ne(pd.Timedelta(minutes=15)).sum()
        ),
        "missing_targets": int(raw_generation["measured"].isna().sum()),
        "negative_targets": int(raw_generation["measured"].dropna().lt(0).sum()),
        "target_above_capacity": int(
            raw_generation["measured"].gt(raw_generation["monitoredcapacity"]).sum()
        ),
        "regions": raw_generation["region"].dropna().unique().tolist(),
        "resolution_codes": raw_generation["resolutioncode"].dropna().unique().tolist(),
    },
    name="value",
)

if generation_audit["duplicate_timestamps"] or generation_audit["unexpected_15_minute_steps"]:
    raise ValueError("Elia timestamps are duplicated or not continuously spaced at 15 minutes.")
if generation_audit["negative_targets"] or generation_audit["target_above_capacity"]:
    raise ValueError("Elia generation contains an impossible physical range.")
if generation_audit["regions"] != ["Belgium"]:
    raise ValueError("Expected only the Belgium aggregate.")
if generation_audit["resolution_codes"] != ["PT15M"]:
    raise ValueError("Expected only PT15M generation intervals.")

generation_audit.to_frame()

,value
rows,70176
columns,7
first_timestamp_utc,2024-01-01 00:00:00+00:00
last_timestamp_utc,2025-12-31 23:45:00+00:00
duplicate_timestamps,0
unexpected_15_minute_steps,0
missing_targets,288
negative_targets,0
target_above_capacity,0
regions,[Belgium]


In [4]:
# Representative source rows; these are not model-ready yet.
raw_generation.head(8)

,datetime,resolutioncode,region,measured,dayaheadforecast,monitoredcapacity,timestamp
0,2024-01-01T00:00:00+00:00,PT15M,Belgium,0.0,0.0,8788.791,2024-01-01 00:00:00+00:00
1,2024-01-01T00:15:00+00:00,PT15M,Belgium,0.0,0.0,8788.791,2024-01-01 00:15:00+00:00
2,2024-01-01T00:30:00+00:00,PT15M,Belgium,0.0,0.0,8788.791,2024-01-01 00:30:00+00:00
3,2024-01-01T00:45:00+00:00,PT15M,Belgium,0.0,0.0,8788.791,2024-01-01 00:45:00+00:00
4,2024-01-01T01:00:00+00:00,PT15M,Belgium,0.0,0.0,8788.791,2024-01-01 01:00:00+00:00
5,2024-01-01T01:15:00+00:00,PT15M,Belgium,0.0,0.0,8788.791,2024-01-01 01:15:00+00:00
6,2024-01-01T01:30:00+00:00,PT15M,Belgium,0.0,0.0,8788.791,2024-01-01 01:30:00+00:00
7,2024-01-01T01:45:00+00:00,PT15M,Belgium,0.0,0.0,8788.791,2024-01-01 01:45:00+00:00


### Inspect the missing-target pattern

Missing target intervals must be distinguished from zero generation. The check below confirms that the source nulls follow a recurring local-clock pattern. Neighboring values and Elia's forecast are shown only as audit evidence; they are **not** used to invent the missing target.

In [5]:
missing_target = raw_generation.loc[raw_generation["measured"].isna()].copy()
missing_target["timestamp_local"] = missing_target["timestamp"].dt.tz_convert(
    "Europe/Brussels"
)
missing_target["previous_measured_mw"] = raw_generation["measured"].shift(1).loc[
    missing_target.index
]
missing_target["next_measured_mw"] = raw_generation["measured"].shift(-1).loc[
    missing_target.index
]

missing_target_audit = pd.Series(
    {
        "missing_quarters": len(missing_target),
        "first_missing_utc": missing_target["timestamp"].min(),
        "last_missing_utc": missing_target["timestamp"].max(),
        "local_clock_values": missing_target["timestamp_local"].dt.strftime("%H:%M").unique().tolist(),
        "nonzero_previous_neighbors": int(missing_target["previous_measured_mw"].ne(0).sum()),
        "nonzero_next_neighbors": int(missing_target["next_measured_mw"].ne(0).sum()),
        "nonzero_day_ahead_values": int(missing_target["dayaheadforecast"].ne(0).sum()),
    },
    name="value",
)
if missing_target_audit["local_clock_values"] != ["23:45"]:
    raise ValueError("The reviewed missing-target pattern has changed.")

missing_target_audit.to_frame()

,value
missing_quarters,288
first_missing_utc,2024-01-01 22:45:00+00:00
last_missing_utc,2024-10-15 21:45:00+00:00
local_clock_values,[23:45]
nonzero_previous_neighbors,0
nonzero_next_neighbors,0
nonzero_day_ahead_values,0


### Convert quarter-hour generation to hourly targets

For target interval `[T, T+1)`, hourly mean power is the arithmetic mean of the four equal 15-minute estimates. Numerically, this is also the interval energy in MWh because `sum(MW × 0.25 h)` equals the four-quarter mean.

An hour with fewer than four non-null target quarters receives `complete_target_hour=False` and a missing hourly target. It is not silently labeled as zero.

In [6]:
generation_hourly, elia_cleaning_audit = load_elia_hourly(ELIA_DATA_PATH)

generation_cleaning_summary = pd.DataFrame(
    [
        {"stage": "Raw 15-minute rows", "rows": elia_cleaning_audit["raw_quarter_hour_rows"]},
        {"stage": "Raw non-null targets", "rows": elia_cleaning_audit["raw_quarter_hour_rows"] - elia_cleaning_audit["raw_missing_targets"]},
        {"stage": "Hourly intervals", "rows": elia_cleaning_audit["hourly_rows"]},
        {"stage": "Complete four-quarter target hours", "rows": elia_cleaning_audit["complete_target_hours"]},
        {"stage": "Incomplete target hours retained with a flag", "rows": int((~generation_hourly["complete_target_hour"]).sum())},
    ]
)
if elia_cleaning_audit != dataset_manifest["audit"]["elia"]:
    raise ValueError("Recomputed Elia audit differs from the frozen dataset manifest.")

generation_cleaning_summary

,stage,rows
0,Raw 15-minute rows,70176
1,Raw non-null targets,69888
2,Hourly intervals,17544
3,Complete four-quarter target hours,17256
4,Incomplete target hours retained with a flag,288


In [7]:
# Example of the conservative rule around the first incomplete target hour.
generation_hourly.loc[
    generation_hourly["timestamp"].between(
        pd.Timestamp("2024-01-01T21:00:00Z"),
        pd.Timestamp("2024-01-01T23:00:00Z"),
    ),
    [
        "timestamp",
        "power_generation_mw",
        "target_quarters",
        "source_quarters",
        "complete_target_hour",
        "monitored_capacity_mw",
    ],
]

,timestamp,power_generation_mw,target_quarters,source_quarters,complete_target_hour,monitored_capacity_mw
21,2024-01-01 21:00:00+00:00,0.0,4,4,True,8788.791
22,2024-01-01 22:00:00+00:00,NaN,3,4,False,8788.791
23,2024-01-01 23:00:00+00:00,0.0,4,4,True,8788.791


## 4. Clean the archived weather forecasts

The archive contains 27 immutable JSON responses: nine Belgian grid points for 2024, 2025, and January 1, 2026. The shared loader performs the following deterministic checks and treatments:

1. Verify every file's byte size and SHA-256 hash.
2. Validate array lengths, field names, units, GMT/UTC offset, point IDs, uniqueness, and hourly cadence.
3. Parse temperature, cloud cover, and shortwave radiation as numeric values.
4. Flag source cloud-cover values outside 0–100% but within the reviewed -1% to 101% rounding range, then clip them to 0–100%.
5. Reject negative shortwave radiation.
6. Preserve null weather values; do not interpolate them.
7. Align valid time `T+1` to target interval `[T, T+1)`.
8. Pivot the nine points to columns and add grid mean and population standard deviation.

In [8]:
weather_source_summary = pd.Series(
    {
        "response_files": weather_manifest["summary"]["files"],
        "grid_points": weather_manifest["requested_points"],
        "raw_point_hour_rows": weather_manifest["summary"]["point_hour_rows"],
        "temperature_nulls": weather_manifest["summary"]["null_counts"]["temperature_2m_previous_day1"],
        "cloud_cover_nulls": weather_manifest["summary"]["null_counts"]["cloud_cover_previous_day1"],
        "radiation_nulls": weather_manifest["summary"]["null_counts"]["shortwave_radiation_previous_day1"],
        "cloud_values_outside_0_100": weather_manifest["summary"]["range_warning_counts"]["cloud_cover_outside_0_100"],
        "negative_radiation_values": weather_manifest["summary"]["range_warning_counts"]["negative_shortwave_radiation"],
        "model": weather_manifest["model"],
        "forecast_offset": weather_manifest["forecast_offset"],
        "timezone": weather_manifest["timezone"],
    },
    name="value",
)
weather_source_summary.to_frame()

,value
response_files,27
grid_points,9
raw_point_hour_rows,158112
temperature_nulls,3996
cloud_cover_nulls,3996
radiation_nulls,3996
cloud_values_outside_0_100,125
negative_radiation_values,0
model,gfs_global
forecast_offset,previous_day1


In [9]:
weather_wide, weather_feature_columns, weather_cleaning_audit = load_weather_wide(
    WEATHER_ROOT, WEATHER_MANIFEST_PATH, GRID_CONFIG_PATH
)

weather_cleaning_summary = pd.Series(
    {
        "point_hour_rows_verified": weather_cleaning_audit["point_hour_rows"],
        "unique_aligned_hours": weather_cleaning_audit["unique_valid_times"],
        "spatial_weather_features": weather_cleaning_audit["spatial_weather_features"],
        "aggregate_weather_features": weather_cleaning_audit["aggregate_weather_features"],
        "cloud_values_clipped": weather_cleaning_audit["cloud_values_clipped"],
        "duplicate_aligned_timestamps": int(weather_wide["timestamp"].duplicated().sum()),
        "weather_feature_columns": len(weather_feature_columns),
        "first_aligned_target_time_utc": weather_wide["timestamp"].min(),
        "last_aligned_target_time_utc": weather_wide["timestamp"].max(),
    },
    name="value",
)
if weather_cleaning_audit != dataset_manifest["audit"]["weather"]:
    raise ValueError("Recomputed weather audit differs from the frozen dataset manifest.")

weather_cleaning_summary.to_frame()

,value
point_hour_rows_verified,158112
unique_aligned_hours,17568
spatial_weather_features,27
aggregate_weather_features,6
cloud_values_clipped,125
duplicate_aligned_timestamps,0
weather_feature_columns,33
first_aligned_target_time_utc,2023-12-31 23:00:00+00:00
last_aligned_target_time_utc,2026-01-01 22:00:00+00:00


In [10]:
# A compact view of the cleaned wide weather table.
weather_preview_columns = [
    "timestamp",
    "temperature_2m_c__grid_mean",
    "temperature_2m_c__grid_std",
    "cloud_cover_pct__grid_mean",
    "cloud_cover_pct__grid_std",
    "shortwave_radiation_w_m2__grid_mean",
    "shortwave_radiation_w_m2__grid_std",
]
first_complete_weather_row = weather_wide[weather_feature_columns].notna().all(axis=1).idxmax()
weather_wide.loc[first_complete_weather_row : first_complete_weather_row + 4, weather_preview_columns]

,timestamp,temperature_2m_c__grid_mean,temperature_2m_c__grid_std,cloud_cover_pct__grid_mean,cloud_cover_pct__grid_std,shortwave_radiation_w_m2__grid_mean,shortwave_radiation_w_m2__grid_std
444,2024-01-19 11:00:00+00:00,0.922222,1.621918,8.666667,2.624669,301.777778,10.674765
445,2024-01-19 12:00:00+00:00,1.233333,1.452966,8.888889,2.514157,296.111111,9.596810
446,2024-01-19 13:00:00+00:00,1.111111,1.333704,7.888889,4.817663,243.555556,7.889671
447,2024-01-19 14:00:00+00:00,0.022222,1.546881,6.777778,3.046958,152.666667,6.733003
448,2024-01-19 15:00:00+00:00,-2.144444,2.207619,6.666667,4.216370,44.000000,5.120764


## 5. Align generation and weather

The join is one-to-one on the aligned UTC target timestamp. Rows remain present even when the target or weather is incomplete, allowing the notebook to count and explain exclusions before filtering.

In [11]:
aligned = generation_hourly.merge(
    weather_wide, on="timestamp", how="left", validate="one_to_one"
)
aligned["weather_complete"] = aligned[weather_feature_columns].notna().all(axis=1)
aligned["target_and_weather_complete"] = (
    aligned["complete_target_hour"]
    & aligned["power_generation_mw"].notna()
    & aligned["weather_complete"]
)

alignment_summary = (
    aligned.assign(year=aligned["timestamp"].dt.year)
    .groupby("year", as_index=False)
    .agg(
        source_hours=("timestamp", "size"),
        complete_target_hours=("complete_target_hour", "sum"),
        complete_weather_hours=("weather_complete", "sum"),
        complete_target_and_weather_hours=("target_and_weather_complete", "sum"),
    )
)
if int(alignment_summary["complete_target_and_weather_hours"].sum()) != 16831:
    raise ValueError("The reviewed target/weather alignment count has changed.")

alignment_summary

,year,source_hours,complete_target_hours,complete_weather_hours,complete_target_and_weather_hours
0,2024,8784,8496,8341,8071
1,2025,8760,8760,8760,8760


## 6. Reconstruct model-eligible rows

Generation lags and temporal columns are deterministic features rather than learned preprocessing. They are reconstructed here only to show how missing source values propagate into the final eligibility filter. Lags are created on the continuous hourly table **before** any rows are removed, so `lag_1h` always means the previous clock hour—not the previous retained row.

In [12]:
target_column = training_config["target_column"]
table_with_lags, lag_columns = add_generation_lags(
    generation_hourly, target_column, training_config["generation_lags_hours"]
)
prepared_table = table_with_lags.merge(
    weather_wide, on="timestamp", how="left", validate="one_to_one"
)
prepared_table, temporal_columns = add_temporal_features(prepared_table)
prepared_table["split"] = assign_splits(prepared_table, training_config["splits"])

feature_columns = [
    "monitored_capacity_mw",
    *weather_feature_columns,
    *temporal_columns,
    *lag_columns,
]
eligibility_checks = pd.DataFrame(
    {
        "configured_split": prepared_table["split"].notna(),
        "complete_target_hour": prepared_table["complete_target_hour"],
        "target_present": prepared_table[target_column].notna(),
        "weather_complete": prepared_table[weather_feature_columns].notna().all(axis=1),
        "lag_1h_present": prepared_table[lag_columns[0]].notna(),
        "lag_24h_present": prepared_table[lag_columns[1]].notna(),
        "lag_168h_present": prepared_table[lag_columns[2]].notna(),
    }
)
eligible = eligibility_checks.all(axis=1)
cleaned_dataset = prepared_table.loc[
    eligible, ["timestamp", "split", target_column, *feature_columns]
].copy()
cleaned_dataset = cleaned_dataset.sort_values("timestamp", kind="stable").reset_index(drop=True)

exclusion_rows = []
for split_name in ["train", "validation", "test"]:
    split_mask = prepared_table["split"].eq(split_name)
    split_checks = eligibility_checks.loc[split_mask]
    exclusion_rows.append(
        {
            "split": split_name,
            "configured_hours": int(split_mask.sum()),
            "incomplete_target_hours": int((~split_checks["complete_target_hour"]).sum()),
            "missing_weather_hours": int((~split_checks["weather_complete"]).sum()),
            "missing_lag_1h_hours": int((~split_checks["lag_1h_present"]).sum()),
            "missing_lag_24h_hours": int((~split_checks["lag_24h_present"]).sum()),
            "missing_lag_168h_hours": int((~split_checks["lag_168h_present"]).sum()),
            "rows_excluded_after_combining_rules": int((~eligible.loc[split_mask]).sum()),
            "rows_retained": int(eligible.loc[split_mask].sum()),
        }
    )
exclusion_summary = pd.DataFrame(exclusion_rows)
exclusion_summary

,split,configured_hours,incomplete_target_hours,missing_weather_hours,missing_lag_1h_hours,missing_lag_24h_hours,missing_lag_168h_hours,rows_excluded_after_combining_rules,rows_retained
0,train,6576,273,443,274,296,434,954,5622
1,validation,2208,15,0,15,16,22,37,2171
2,test,8760,0,0,0,0,0,0,8760


In [13]:
split_summary = pd.DataFrame(
    [
        {
            "split": split_name,
            "rows": len(part),
            "columns": len(part.columns),
            "model_features": len(feature_columns),
            "first_timestamp_utc": part["timestamp"].min(),
            "last_timestamp_utc": part["timestamp"].max(),
            "missing_cells": int(part.isna().sum().sum()),
            "duplicate_timestamps": int(part["timestamp"].duplicated().sum()),
        }
        for split_name in ["train", "validation", "test"]
        for part in [cleaned_dataset.loc[cleaned_dataset["split"].eq(split_name)]]
    ]
)
split_summary

,split,rows,columns,model_features,first_timestamp_utc,last_timestamp_utc,missing_cells,duplicate_timestamps
0,train,5622,47,44,2024-01-19 11:00:00+00:00,2024-09-30 23:00:00+00:00,0,0
1,validation,2171,47,44,2024-10-01 00:00:00+00:00,2024-12-31 23:00:00+00:00,0,0
2,test,8760,47,44,2025-01-01 00:00:00+00:00,2025-12-31 23:00:00+00:00,0,0


### Final integrity checks

These checks validate schema, finite values, physical target bounds, timestamp uniqueness, chronological ordering, split row counts, and prepared-file hashes. Hashes are checked without loading or summarizing the sealed test target.

In [14]:
expected_columns = ["timestamp", "split", target_column, *feature_columns]
if list(cleaned_dataset.columns) != expected_columns:
    raise ValueError("Cleaned dataset columns differ from the configured schema.")
if feature_columns != dataset_manifest["feature_columns"]:
    raise ValueError("Feature order differs from the frozen dataset manifest.")
if target_column in feature_columns or {"timestamp", "split"} & set(feature_columns):
    raise ValueError("Target or metadata leakage detected in model features.")
if cleaned_dataset.isna().any().any():
    raise ValueError("Cleaned model-eligible data still contains missing cells.")
numeric = cleaned_dataset[[target_column, *feature_columns]].to_numpy(dtype="float64")
if not np.isfinite(numeric).all():
    raise ValueError("Cleaned model-eligible data contains a non-finite value.")
if cleaned_dataset["timestamp"].duplicated().any():
    raise ValueError("Cleaned model-eligible data contains duplicate timestamps.")
if not cleaned_dataset["timestamp"].is_monotonic_increasing:
    raise ValueError("Cleaned model-eligible data is not chronological.")
if cleaned_dataset[target_column].lt(0).any():
    raise ValueError("Cleaned target contains negative generation.")
if cleaned_dataset[target_column].gt(cleaned_dataset["monitored_capacity_mw"]).any():
    raise ValueError("Cleaned target exceeds monitored capacity.")

manifest_rows = {entry["split"]: entry["rows"] for entry in dataset_manifest["files"]}
actual_rows = cleaned_dataset["split"].value_counts().to_dict()
if actual_rows != manifest_rows:
    raise ValueError(f"Reconstructed split counts differ: {actual_rows} != {manifest_rows}")

prepared_file_checks = []
for entry in dataset_manifest["files"]:
    path = PREPARED_ROOT / entry["file"]
    prepared_file_checks.append(
        {
            "split": entry["split"],
            "file": str(path.relative_to(ROOT)),
            "rows_expected": entry["rows"],
            "bytes_match": path.stat().st_size == entry["bytes"],
            "sha256_match": sha256_file(path) == entry["sha256"],
        }
    )
prepared_file_checks = pd.DataFrame(prepared_file_checks)
if not prepared_file_checks[["bytes_match", "sha256_match"]].all().all():
    raise ValueError("A prepared split file differs from its frozen manifest.")

prepared_file_checks

,split,file,rows_expected,bytes_match,sha256_match
0,train,data/processed/model_training/train.csv,5622,True,True
1,validation,data/processed/model_training/validation.csv,2171,True,True
2,test,data/processed/model_training/test.csv,8760,True,True


## 7. Cleaning decision log

This compact report distinguishes validation, cleaning, and row-filtering actions. Counts for different exclusion reasons can overlap because one hour may fail more than one requirement.

In [15]:
cleaning_decisions = pd.DataFrame(
    [
        {
            "issue": "Missing Elia target quarters",
            "observed": len(missing_target),
            "treatment": "Keep null; flag the hourly target incomplete",
            "reason": "Missing generation is not proof of zero generation",
        },
        {
            "issue": "Duplicate Elia timestamps",
            "observed": int(raw_generation["timestamp"].duplicated().sum()),
            "treatment": "Reject the source if any are present",
            "reason": "Duplicate intervals make hourly aggregation ambiguous",
        },
        {
            "issue": "Raw weather null cells",
            "observed": int(sum(weather_manifest["summary"]["null_counts"].values())),
            "treatment": "Preserve nulls; later exclude incomplete model rows",
            "reason": "No unreviewed interpolation or backfill",
        },
        {
            "issue": "Cloud cover at -1% or 101%",
            "observed": weather_cleaning_audit["cloud_values_clipped"],
            "treatment": "Flag and clip to the physical 0–100% range",
            "reason": "Reviewed source rounding artifacts",
        },
        {
            "issue": "Negative shortwave radiation",
            "observed": weather_manifest["summary"]["range_warning_counts"]["negative_shortwave_radiation"],
            "treatment": "Reject the source if any are present",
            "reason": "Negative values are invalid for this feature contract",
        },
        {
            "issue": "Rows failing combined target/weather/lag rules",
            "observed": dataset_manifest["audit"]["rows_excluded"],
            "treatment": "Exclude from model-ready splits after auditing",
            "reason": "Final matrix must be finite without target imputation",
        },
        {
            "issue": "Elia day-ahead forecast",
            "observed": "source column present",
            "treatment": "Exclude from model features",
            "reason": "Exact historical issue-time value is unavailable",
        },
    ]
)
cleaning_decisions

,issue,observed,treatment,reason
0,Missing Elia target quarters,288,Keep null; flag the hourly target incomplete,Missing generation is not proof of zero genera...
1,Duplicate Elia timestamps,0,Reject the source if any are present,Duplicate intervals make hourly aggregation am...
2,Raw weather null cells,11988,Preserve nulls; later exclude incomplete model...,No unreviewed interpolation or backfill
3,Cloud cover at -1% or 101%,125,Flag and clip to the physical 0–100% range,Reviewed source rounding artifacts
4,Negative shortwave radiation,0,Reject the source if any are present,Negative values are invalid for this feature c...
5,Rows failing combined target/weather/lag rules,991,Exclude from model-ready splits after auditing,Final matrix must be finite without target imp...
6,Elia day-ahead forecast,source column present,Exclude from model features,Exact historical issue-time value is unavailable


## 8. Optional audit export

The notebook is read-only by default. Set `WRITE_AUDIT = True` only when you intentionally want a local JSON report under `artifacts/data_quality/`. The raw and prepared datasets are never written by this cell. Existing audit files are not replaced unless `OVERWRITE_AUDIT` is also enabled.

In [16]:
cleaning_audit = {
    "schema_version": "belgium-solar-cleaning-audit-v1",
    "source_schema_version": dataset_manifest["schema_version"],
    "raw_elia_rows": int(elia_cleaning_audit["raw_quarter_hour_rows"]),
    "raw_elia_missing_targets": int(elia_cleaning_audit["raw_missing_targets"]),
    "complete_hourly_targets": int(elia_cleaning_audit["complete_target_hours"]),
    "raw_weather_point_hours": int(weather_cleaning_audit["point_hour_rows"]),
    "cloud_values_clipped": int(weather_cleaning_audit["cloud_values_clipped"]),
    "complete_target_weather_hours": int(aligned["target_and_weather_complete"].sum()),
    "model_features": len(feature_columns),
    "rows_excluded": int(dataset_manifest["audit"]["rows_excluded"]),
    "split_rows": {name: int(count) for name, count in actual_rows.items()},
    "target_imputed": False,
    "weather_imputed": False,
    "test_target_summarized": False,
}

WRITE_AUDIT = False
OVERWRITE_AUDIT = False
AUDIT_OUTPUT = ROOT / "artifacts/data_quality/cleaning_audit.json"
if WRITE_AUDIT:
    AUDIT_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    if AUDIT_OUTPUT.exists() and not OVERWRITE_AUDIT:
        raise FileExistsError(
            f"Refusing to replace {AUDIT_OUTPUT}; set OVERWRITE_AUDIT=True intentionally."
        )
    AUDIT_OUTPUT.write_text(
        json.dumps(cleaning_audit, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    print(f"Wrote {AUDIT_OUTPUT.relative_to(ROOT)}")
else:
    print("Audit export disabled; no files were written.")

cleaning_audit

Audit export disabled; no files were written.


{'schema_version': 'belgium-solar-cleaning-audit-v1',
 'source_schema_version': 'belgium-solar-training-v1',
 'raw_elia_rows': 70176,
 'raw_elia_missing_targets': 288,
 'complete_hourly_targets': 17256,
 'raw_weather_point_hours': 158112,
 'cloud_values_clipped': 125,
 'complete_target_weather_hours': 16831,
 'model_features': 44,
 'rows_excluded': 991,
 'split_rows': {'test': 8760, 'train': 5622, 'validation': 2171},
 'target_imputed': False,
 'weather_imputed': False,
 'test_target_summarized': False}

## Result

The frozen sources have been checksummed, parsed, validated, cleaned, aligned, and reconstructed into the documented chronological splits. The final model-ready table has 44 ordered features, no missing/non-finite cells, no duplicate timestamps, and no target or metadata columns in the feature matrix.

Important limitations remain:

- The recurring 23:45 Europe/Brussels missing target is preserved as missing, not converted to zero.
- Early-2024 archived weather is unavailable and is not interpolated.
- Historical generation lags use final archived estimates; live availability and revisions still require forward validation.
- Complete-case filtering leaves timestamp gaps in 2024; models must use the timestamp and explicit lags rather than treating adjacent retained rows as adjacent hours.
- The untouched 2025 target must not guide cleaning changes, feature choices, or hyperparameters.

To regenerate the official prepared CSV files from these same rules, run from the repository root:

```bash
uv run python -m scripts.prepare_training_data --force
```

Use `--force` only when intentionally replacing derived files from the unchanged frozen inputs. The next step is exploratory analysis on training and validation data, followed by persistence baselines.